In [ ]:
import os

import geopandas as gpd
import overpass
import requests as r

In [2]:
def fetch_european_countries_via_api():
    # REST Countries API returns all nations with region info
    resp = r.get("https://restcountries.com/v3.1/region/europe")
    resp.raise_for_status()
    data = resp.json()

    # Extract the common name of each country
    return data

def get_street_query(
    extent_country: str, street_type_depth: int = 2, timeout: int = 240
) -> str:
    """Generate an Overpass API query to fetch motorways within a specified country extent.

    Args:
        extent_country (str): The country code to define the extent. (e.g., "DE" for Germany)
        street_type_depth (int): The depth of street types to include. Default is 2. Find more information at https://wiki.openstreetmap.org/wiki/Key:highway.
        timeout (int): The timeout for the Overpass API request in seconds. Default is 240.

    Returns:
        str: The generated Overpass API query string.
    """
    query = f"""
    area["ISO3166-1"="{extent_country}"]->.searchArea;
    (
        way(area.searchArea){get_street_filter_string(street_type_depth)};
    );
    """
    return query


def get_street_filter_string(street_type_depth: int) -> str:
    """Generate a filter string for street types based on the specified depth.

    Args:
        street_type_depth (int): The depth of street types to include. Find more information at https://wiki.openstreetmap.org/wiki/Key:highway.

    Returns:
        str: The generated filter string for street types.
    """
    street_types = [
        "motorway",
        "trunk",
        "primary",
        "secondary",
        "tertiary",
        "unclassified",
        "residential",
        "service",
        "living_street",
        "pedestrian",
        "track",
        "path",
    ]
    if street_type_depth < 1 or street_type_depth > len(street_types):
        raise ValueError(f"street_type_depth must be between 1 and {len(street_types)}")
    selected_street_types = street_types[:street_type_depth]
    filter_string = '["highway"~"' + "|".join(selected_street_types) + '"]'
    return filter_string

In [3]:
api = overpass.API(timeout=300)

In [4]:
europe = fetch_european_countries_via_api()

In [ ]:
depth = 3

In [17]:
for country in europe:
    try:
        if os.path.exists(f"../data/streets/depth_{depth}/streets_{country['cca2']}_depth{depth}.parquet"):
            print(f"Streets for {country['cca2']} already exist, skipping.")
            continue
        query = get_street_query(extent_country=country["cca2"], street_type_depth=depth)
        print(f"Fetching streets for {country['name']['common']} ({country['cca2']})...")
        response = api.get(query, verbosity="body geom")
        print(response)
        if len(response["features"]) == 0:
            print(f"No streets found for {country['name']['common']}, at depth {depth}, skipping.")
            continue
        gdf = gpd.GeoDataFrame.from_features(response["features"], crs="EPSG:4326")
        gdf.to_parquet(f"../data/streets/depth_{depth}/streets_{country['cca2']}_depth{depth}.parquet")
    except Exception as e:
        print(f"Error fetching streets for {country['name']['common']}: {e}")
        continue

Streets for IT already exist, skipping.
Streets for JE already exist, skipping.
Streets for MK already exist, skipping.
Streets for LV already exist, skipping.
Streets for EE already exist, skipping.
Streets for BY already exist, skipping.
Streets for CH already exist, skipping.
Streets for XK already exist, skipping.
Fetching streets for Liechtenstein (LI)...
Error fetching streets for Liechtenstein: 25
Streets for BE already exist, skipping.
Streets for IS already exist, skipping.
Streets for SI already exist, skipping.
Fetching streets for Åland Islands (AX)...
{'type': 'FeatureCollection', 'features': []}
No streets found for Åland Islands, at depth 2, skipping.
Streets for UA already exist, skipping.
Streets for GG already exist, skipping.
Streets for CZ already exist, skipping.
Fetching streets for Vatican City (VA)...
{'type': 'FeatureCollection', 'features': []}
No streets found for Vatican City, at depth 2, skipping.
Streets for ME already exist, skipping.
Streets for AT alrea

KeyboardInterrupt: 